In [2]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [3]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [4]:
from load_network import load_Mix
from train import WarmUpCosine, CustomWeightDecaySGD
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_cla import load_wb_if_exists
from network import HierarchicalConvEmbedding, MLP, MixerBlock
from evaluate_cla import evalu_stream_main_selected, per_group_ovr_accuracy, evalu_select

In [5]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [6]:
model=load_Mix()

In [7]:
model.evaluate(x_test,y_test_onehot)

125/125 [==============================] - 13s 22ms/step - loss: 0.2594 - accuracy: 0.9277


[0.25943368673324585, 0.9277499914169312]

In [6]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("reg_dense_0").output,
    outputs=model.output
)

In [7]:
l_label = [1,2,3,4,5,6,7,8,9,12]

In [8]:
layer_list = [model.layers[l].name for l in l_label]

In [9]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_1/Mix-8/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_1/Mix-8/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_1/Mix-8/layer_cache/salt/"+str(i))
    save_layer_outputs_and_labels(model, x_move, y_train, layer_list, save_dir="D:/Data_1/Mix-8/layer_cache/move/"+str(i))
    save_layer_outputs_and_labels(model, x_occ, y_train, layer_list, save_dir="D:/Data_1/Mix-8/layer_cache/occ/"+str(i))

[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/base
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/gauss/0
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/salt/0
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/move/0
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/occ/0
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/gauss/1
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/salt/1
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/move/1
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/occ/1
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/gauss/2
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/salt/2
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/move/2
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/occ/2
[SKIP] All layer files alre

In [10]:
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "Mix_pgd.npy")
    x_attack = np.load(ATTACK_DIR)
    save_layer_outputs_and_labels(model, x_attack, y_train, layer_list, save_dir="D:/Data_1/Mix-8/layer_cache/attack/"+str(i))

[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/attack/0
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/attack/1
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/attack/2
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/attack/3
[SKIP] All layer files already exist in D:/Data_1/Mix-8/layer_cache/attack/4


In [11]:
CACHE_DIR = "./Mix-8/w_and_b_cache"

In [12]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_1/Mix-8/layer_cache/base")

In [13]:
NOISE_dir='./noise/'
ATTACK_dir=Noise_dir='./attack/'

In [14]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_1/Mix-8/layer_cache/base")

[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]


In [15]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_1/Mix-8/layer_cache/base")
base_final = per_group_ovr_accuracy(model, x_train, y_train, select_list)
base = np.concatenate((base,base_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.67054499 0.54904701 0.56432508 0.5803611 ]
Layer 1
accuracy: [0.66532222 0.58963988 0.65979304 0.63518937]
Layer 2
accuracy: [0.69255417 0.65396237 0.74467741 0.71023065]
Layer 3
accuracy: [0.6966146  0.70136251 0.7758042  0.75603925]
Layer 4
accuracy: [0.74123395 0.74971503 0.82881673 0.77707667]
Layer 5
accuracy: [0.76347812 0.78548219 0.85892831 0.82324419]
Layer 6
accuracy: [0.85878514 0.86328    0.914781   0.87675697]
Layer 7
accuracy: [0.93614824 0.94243708 0.96843545 0.94551493]
Layer 8
accuracy: [0.98752783 0.98754987 0.99020688 0.9880157 ]
Layer 9
accuracy: [0.9910849  0.98774615 0.99129622 0.98796501]


In [16]:
base

array([[0.67054499, 0.54904701, 0.56432508, 0.5803611 ],
       [0.66532222, 0.58963988, 0.65979304, 0.63518937],
       [0.69255417, 0.65396237, 0.74467741, 0.71023065],
       [0.6966146 , 0.70136251, 0.7758042 , 0.75603925],
       [0.74123395, 0.74971503, 0.82881673, 0.77707667],
       [0.76347812, 0.78548219, 0.85892831, 0.82324419],
       [0.85878514, 0.86328   , 0.914781  , 0.87675697],
       [0.93614824, 0.94243708, 0.96843545, 0.94551493],
       [0.98752783, 0.98754987, 0.99020688, 0.9880157 ],
       [0.9910849 , 0.98774615, 0.99129622, 0.98796501],
       [1.        , 1.        , 1.        , 1.        ]])

In [17]:
attack = np.zeros((len(layer_list),4))
for i in range(5):
    attack += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Mix-8/layer_cache/attack/"+str(i))
attack = attack/5
attack_final = np.zeros(4)
for i in range(5):
    DIR = ATTACK_dir + str(i) + '/Mix_pgd.npy'
    x_noise = np.load(DIR)
    attack_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
attack_final = attack_final/5
attack = np.concatenate((attack,attack_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.67507406 0.52133972 0.56766851 0.5456443 ]
Layer 1
accuracy: [0.67173565 0.53658484 0.64758489 0.51560412]
Layer 2
accuracy: [0.63667872 0.56189081 0.65572795 0.51354263]
Layer 3
accuracy: [0.6271406  0.59765978 0.67688614 0.52171606]
Layer 4
accuracy: [0.6276517  0.589457   0.69243081 0.48097586]
Layer 5
accuracy: [0.58798616 0.55312514 0.67438039 0.534925  ]
Layer 6
accuracy: [0.49424713 0.51812531 0.58417174 0.50168582]
Layer 7
accuracy: [0.40425827 0.56782011 0.53160809 0.47119957]
Layer 8
accuracy: [0.27703108 0.37756728 0.45171612 0.24424119]
Layer 9
accuracy: [0.2270845  0.20584213 0.42519039 0.0767906 ]
Layer 0
accuracy: [0.67437194 0.51902793 0.56906884 0.5471242 ]
Layer 1
accuracy: [0.67560016 0.53477603 0.6473352  0.51269673]
Layer 2
accuracy: [0.64206231 0.55949866 0.65225986 0.51166259]
Layer 3
accuracy: [0.63262882 0.59691504 0.67527871 0.52344348]
Layer 4
accuracy: [0.63232191 0.58494753 0.68787885 0.47949191]
Layer 5
accuracy: [0.58873349 0.55149304

In [18]:
attack

array([[0.67421073, 0.52078361, 0.56935944, 0.54605971],
       [0.67368591, 0.53549073, 0.64787276, 0.51423548],
       [0.63982148, 0.56118462, 0.65649803, 0.50994045],
       [0.62942537, 0.59805842, 0.67669112, 0.52210026],
       [0.62903951, 0.58944503, 0.69129105, 0.47904865],
       [0.58938762, 0.55545885, 0.67136713, 0.53362991],
       [0.49911574, 0.52136563, 0.58409742, 0.50191862],
       [0.40361076, 0.5720534 , 0.53029398, 0.46611484],
       [0.27838784, 0.37494005, 0.45073444, 0.24032771],
       [0.22836549, 0.20842924, 0.42627931, 0.07511914],
       [0.22915   , 0.20805   , 0.42545   , 0.0753    ]])

In [19]:
gauss = np.zeros((len(layer_list),4))
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Mix-8/layer_cache/gauss/"+str(i))
gauss = gauss/10
gauss_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/gauss.npy'
    x_noise = np.load(DIR)
    gauss_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.69617503 0.29254709 0.68395226 0.39986497]
Layer 1
accuracy: [0.74954477 0.456779   0.74975244 0.54348905]
Layer 2
accuracy: [0.75003627 0.45227209 0.75000194 0.5622893 ]
Layer 3
accuracy: [0.74978948 0.66814273 0.75000194 0.67647454]
Layer 4
accuracy: [0.74978948 0.64408891 0.75000194 0.6499988 ]
Layer 5
accuracy: [0.75003627 0.66439198 0.75000194 0.7096463 ]
Layer 6
accuracy: [0.75449727 0.77111072 0.75000194 0.67836242]
Layer 7
accuracy: [0.75475089 0.77663594 0.75000194 0.63144229]
Layer 8
accuracy: [0.74503455 0.77358212 0.74975441 0.65280645]
Layer 9
accuracy: [0.61365771 0.77853495 0.748733   0.40543764]
Layer 0
accuracy: [0.70220805 0.29238253 0.68519708 0.39987523]
Layer 1
accuracy: [0.74954231 0.45348478 0.74950491 0.54544514]
Layer 2
accuracy: [0.74902591 0.45145019 0.75000194 0.56097827]
Layer 3
accuracy: [0.75003627 0.6691063  0.75000194 0.66906075]
Layer 4
accuracy: [0.74952896 0.65026865 0.75000194 0.63911649]
Layer 5
accuracy: [0.74976666 0.66334179

In [20]:
gauss

array([[0.70057709, 0.29065707, 0.68562605, 0.39734602],
       [0.74941094, 0.45582216, 0.74955461, 0.54892215],
       [0.74975924, 0.44925983, 0.75000194, 0.56336773],
       [0.74981258, 0.6649308 , 0.75000194, 0.67287702],
       [0.74976156, 0.64535071, 0.75000194, 0.64605943],
       [0.75003241, 0.66237966, 0.75000194, 0.70933067],
       [0.75418385, 0.77174247, 0.75000194, 0.67239681],
       [0.75172558, 0.7773449 , 0.75000194, 0.62858966],
       [0.74233985, 0.77204725, 0.74992625, 0.64775756],
       [0.6105031 , 0.78105463, 0.74962924, 0.40411881],
       [0.670125  , 0.78285   , 0.750875  , 0.52822499]])

In [21]:
salt = np.zeros((len(layer_list),4))
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Mix-8/layer_cache/salt/"+str(i))
salt = salt/10
salt_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/salt.npy'
    x_noise = np.load(DIR)
    salt_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.60319397 0.2594117  0.53839408 0.27926257]
Layer 1
accuracy: [0.73469525 0.37318617 0.74926125 0.47368201]
Layer 2
accuracy: [0.74418498 0.44494016 0.75100053 0.55076465]
Layer 3
accuracy: [0.74953285 0.64980806 0.75049313 0.68546193]
Layer 4
accuracy: [0.74930252 0.66321865 0.75000194 0.67901468]
Layer 5
accuracy: [0.75134362 0.68425697 0.75000194 0.76450578]
Layer 6
accuracy: [0.74614229 0.81478192 0.7512357  0.7819867 ]
Layer 7
accuracy: [0.71015764 0.82058054 0.75346626 0.76964599]
Layer 8
accuracy: [0.7118218  0.82036893 0.75950208 0.80962653]
Layer 9
accuracy: [0.50119678 0.83382456 0.78263628 0.66456396]
Layer 0
accuracy: [0.60063716 0.25952303 0.53735439 0.27874209]
Layer 1
accuracy: [0.73496514 0.38032928 0.74926125 0.48070533]
Layer 2
accuracy: [0.74325954 0.45758987 0.75000194 0.55637585]
Layer 3
accuracy: [0.74734571 0.64872977 0.75000194 0.68039058]
Layer 4
accuracy: [0.74863931 0.67293881 0.75000194 0.68249034]
Layer 5
accuracy: [0.75000064 0.68249042

In [22]:
salt

array([[0.60489491, 0.25719196, 0.53556398, 0.28026263],
       [0.73627505, 0.37157479, 0.74873375, 0.47569348],
       [0.74297251, 0.45230411, 0.75012817, 0.5564    ],
       [0.74773173, 0.64798852, 0.75007601, 0.68430593],
       [0.74903319, 0.66417618, 0.75002689, 0.68678461],
       [0.74935174, 0.68123279, 0.75002689, 0.7638131 ],
       [0.74746465, 0.80763723, 0.75085968, 0.77325944],
       [0.70734541, 0.82233307, 0.75335839, 0.7662005 ],
       [0.71456862, 0.82065296, 0.75971859, 0.80672773],
       [0.50051168, 0.83609946, 0.7825042 , 0.67157819],
       [0.550675  , 0.84615   , 0.7845    , 0.77905   ]])

In [23]:
move = np.zeros((len(layer_list),4))
for i in range(10):
    move += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Mix-8/layer_cache/move/"+str(i))
move = move/10
move_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/move.npy'
    x_noise = np.load(DIR)
    move_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
move_final = move_final/10
move = np.concatenate((move,move_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.69844178 0.73136466 0.59946488 0.71491968]
Layer 1
accuracy: [0.66932012 0.75023238 0.62777661 0.74769593]
Layer 2
accuracy: [0.69189903 0.75023238 0.64536342 0.74869666]
Layer 3
accuracy: [0.69849102 0.75023238 0.68719224 0.74843412]
Layer 4
accuracy: [0.73694635 0.75023238 0.70968341 0.74819351]
Layer 5
accuracy: [0.74569297 0.75023238 0.68890744 0.74918597]
Layer 6
accuracy: [0.76316934 0.75047346 0.67381926 0.74920717]
Layer 7
accuracy: [0.77794739 0.75048774 0.70510129 0.748223  ]
Layer 8
accuracy: [0.80213807 0.7534291  0.76347479 0.74800208]
Layer 9
accuracy: [0.7980265  0.74749705 0.68121721 0.74519463]
Layer 0
accuracy: [0.69741239 0.73140962 0.59880833 0.71471345]
Layer 1
accuracy: [0.6727748  0.75023238 0.62945365 0.74646465]
Layer 2
accuracy: [0.69310914 0.75023238 0.63870662 0.74845036]
Layer 3
accuracy: [0.69972613 0.75023238 0.68589469 0.74894842]
Layer 4
accuracy: [0.74494216 0.75023238 0.70334493 0.74643001]
Layer 5
accuracy: [0.75629862 0.75023238

In [24]:
move

array([[0.6967633 , 0.73197328, 0.59792739, 0.71623458],
       [0.67232061, 0.75023238, 0.62822592, 0.74705892],
       [0.69526046, 0.75023238, 0.64233526, 0.74900416],
       [0.70047701, 0.75023238, 0.68500455, 0.74808156],
       [0.74195236, 0.75023238, 0.70567177, 0.74638849],
       [0.74871141, 0.75023238, 0.68255496, 0.74830474],
       [0.76320116, 0.75027526, 0.66404243, 0.74759748],
       [0.77673749, 0.75026775, 0.69549007, 0.74827199],
       [0.80177657, 0.75387228, 0.75984711, 0.74819753],
       [0.79410983, 0.74992361, 0.67798988, 0.74695338],
       [0.8058    , 0.747625  , 0.68552499, 0.73152501]])

In [25]:
occ = np.zeros((len(layer_list),4))
for i in range(10):
    occ += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/Mix-8/layer_cache/occ/"+str(i))
occ = occ/10
occ_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/occ.npy'
    x_noise = np.load(DIR)
    occ_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
occ_final = occ_final/10
occ = np.concatenate((occ,occ_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.62553    0.41561144 0.5111212  0.4752635 ]
Layer 1
accuracy: [0.65817154 0.64312957 0.64678834 0.61614601]
Layer 2
accuracy: [0.71345023 0.73331902 0.74644815 0.70900038]
Layer 3
accuracy: [0.7241014  0.74765697 0.76905848 0.73960476]
Layer 4
accuracy: [0.74966199 0.75187094 0.79501158 0.74536613]
Layer 5
accuracy: [0.75699162 0.75464534 0.79977556 0.75932238]
Layer 6
accuracy: [0.7897485  0.77744457 0.82736477 0.74866584]
Layer 7
accuracy: [0.80958443 0.79198164 0.84419677 0.73997427]
Layer 8
accuracy: [0.82130352 0.81193657 0.82724598 0.73062975]
Layer 9
accuracy: [0.82839029 0.81490025 0.77118075 0.67992291]
Layer 0
accuracy: [0.62300135 0.41782859 0.50262036 0.4733078 ]
Layer 1
accuracy: [0.65894113 0.63956247 0.6499451  0.61122636]
Layer 2
accuracy: [0.71383594 0.73312049 0.74995585 0.70135207]
Layer 3
accuracy: [0.72628599 0.74639654 0.77450925 0.73680169]
Layer 4
accuracy: [0.74729504 0.75070823 0.79420694 0.73784195]
Layer 5
accuracy: [0.7535563  0.75424737

In [26]:
occ

array([[0.62550457, 0.41535521, 0.50590829, 0.47262574],
       [0.65817278, 0.64017492, 0.65118903, 0.61118211],
       [0.71368907, 0.73353818, 0.74579246, 0.70537019],
       [0.72617366, 0.74647087, 0.76960397, 0.73818215],
       [0.75072825, 0.7508038 , 0.79194788, 0.74055655],
       [0.75339358, 0.75472884, 0.79779018, 0.75779233],
       [0.78457931, 0.77377456, 0.83097739, 0.74387781],
       [0.80629433, 0.78947145, 0.84392769, 0.73362719],
       [0.82046344, 0.80932119, 0.83598422, 0.72151968],
       [0.82253385, 0.80715541, 0.77731671, 0.67656311],
       [0.85120001, 0.814525  , 0.80324999, 0.69879999]])

In [27]:
SAVE_FILE='Mix-8.pkl'

In [28]:
progress = {"base": base, "attack": attack,"gauss": gauss,"salt": salt,"move": move,"occ": occ}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [29]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=6):
    """
    Input:
        matrix_dict: {name: np.ndarray(m, n)}, 6 matrices total
    Output:
        stats: {
            name: {
                "mean": (m,),
                "min":  (m,),
                "max":  (m,)
            }, ...
        }
    For each component i, compute statistics along axis=1 (across n samples):
        mean[i] = mean(X[i, :])
        min[i]  = min(X[i, :])
        max[i]  = max(X[i, :])
    """
    if check_num is not None and len(matrix_dict) != check_num:
        raise ValueError(f"Expected {check_num} matrices, got {len(matrix_dict)}")

    # shape consistency
    shapes = [np.asarray(v).shape for v in matrix_dict.values()]
    if len(set(shapes)) != 1:
        raise ValueError(f"Matrix shapes inconsistent: {shapes}")

    stats = {}
    for name, X in matrix_dict.items():
        X = np.asarray(X)
        stats[name] = {
            "mean": X.mean(axis=1),
            "std":  X.std(axis=1),
            "min":  X.min(axis=1),
            "max":  X.max(axis=1),
        }
    return stats

In [31]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [32]:
mean_var

{'base': {'mean': array([0.59106954, 0.63748613, 0.70035615, 0.73245514, 0.7742106 ,
         0.8077832 , 0.87840077, 0.94813392, 0.98832507, 0.98952307,
         1.        ]),
  'std': array([0.04720216, 0.02986212, 0.03269225, 0.03422954, 0.03419626,
         0.0364525 , 0.02202064, 0.01219756, 0.0011038 , 0.00167096,
         0.        ]),
  'min': array([0.54904701, 0.58963988, 0.65396237, 0.6966146 , 0.74123395,
         0.76347812, 0.85878514, 0.93614824, 0.98752783, 0.98774615,
         1.        ]),
  'max': array([0.67054499, 0.66532222, 0.74467741, 0.7758042 , 0.82881673,
         0.85892831, 0.914781  , 0.96843545, 0.99020688, 0.99129622,
         1.        ])},
 'attack': {'mean': array([0.57760337, 0.59282122, 0.59186115, 0.60656879, 0.59720606,
         0.58746088, 0.52662435, 0.49301825, 0.33609751, 0.23454829,
         0.2344875 ]),
  'std': array([0.05836187, 0.06897875, 0.05943519, 0.05622927, 0.07727671,
         0.05235904, 0.03427063, 0.06394045, 0.08239027, 0.1253